# 01 - Figure 1: transport mechanism

**Question.** How are player trajectories reduced to a team-centroid
trajectory, segmented into runs, and related to collective order?

| Panel | Construction |
|---|---|
| A | Real 20-second player and centroid trajectories on the calibrated pitch |
| B | Direction-change segmentation of the centroid path |
| C/D | High- and low-polarisation player configurations |

The selected match windows and required caches are shown before the
final plotting function is called.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# Locate the repository before importing its analysis package. This works when
# Jupyter starts from either the repo root or this notebook directory.
_start = Path.cwd()
ROOT = next(
    path for path in (_start, *_start.parents)
    if (path / "analysis" / "levy_paper").is_dir()
    and (path / "requirements.txt").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.levy_paper.util.publication_notebook_utils import (
    PRIMARY_CACHE_SUFFIX,
    cache_path as make_cache_path,
    csv_shapes,
    display_live_or_frozen,
    file_status,
    hazard_support_summary,
    load_processed_cache,
    order_state_summary,
    panel_inventory,
    publication_paths,
    relative_path,
    resolve_data_mode,
    table_inventory,
    transition_row_sum_audit,
    transport_run_summary,
)

PATHS = publication_paths(ROOT)
LEVY_DIR = PATHS["levy_dir"]
DATA_DIR = PATHS["data_dir"]
PRIMARY_CACHE_DIR = PATHS["primary_cache_dir"]
FINAL_FIGURES = PATHS["final_figures"]
SOURCE_DATA = PATHS["source_data"]
SUPPLEMENT = PATHS["supplement"]
CACHE_SUFFIX = PRIMARY_CACHE_SUFFIX

# DATA_MODE options:
#   "auto"     use processed caches when all required files exist;
#              otherwise use tracked reviewer tables/frozen figures
#   "cache"    require processed caches and fail clearly if they are missing
#   "reviewer" use only tracked public artefacts
DATA_MODE = "auto"
BUILD_FIGURE = True
SAVE_FIGURE_OUTPUTS = True
DISPLAY_FROZEN_OUTPUT = True
REBUILD_CACHE_FROM_AWS = False
REFIT_FIGURE4_FIGURE5_MODELS = False
USE_VERSIONED_FINAL_FIGURE4_FIT = True


def rel(path):
    return relative_path(path, ROOT)


def show_file_status(paths):
    return file_status(paths, ROOT)


def show_csv_shapes(paths):
    return csv_shapes(paths, ROOT)


def cache_path(stem):
    return make_cache_path(PRIMARY_CACHE_DIR, stem, CACHE_SUFFIX)


def show_figure(fig, frozen_path, width=1100):
    return display_live_or_frozen(
        fig,
        frozen_path,
        display_frozen=DISPLAY_FROZEN_OUTPUT,
        width=width,
    )

## 1. Data mode and required inputs

In [ ]:
from analysis.levy_paper.scripts import create_real_match_transport_mechanism_4panel as figure1

figure1_inputs = [cache_path("trajectory_long"), cache_path("runs_long"), cache_path("df_pmv")]
resolved_mode = resolve_data_mode(DATA_MODE, figure1_inputs)
cache_ready = resolved_mode == "cache"
print("resolved_data_mode", resolved_mode)
display(show_file_status(figure1_inputs))

## 2. Available match/team trajectories

In [ ]:
games = pd.DataFrame()

if cache_ready:
    games = figure1.available_games(limit=30)
    display(games)
else:
    print("Processed trajectory cache unavailable; reviewer fallback will be used.")

## 3. Selected real-match windows

In [ ]:
selection = pd.DataFrame([
    {"panel": "A", "match_id": figure1.PANEL_A_MATCH_ID, "phase": figure1.PANEL_A_MATCH_PHASE, "team": figure1.PANEL_A_TEAM, "source": figure1.PANEL_A_SOURCE_KEY, "t0": figure1.PANEL_A_T0},
    {"panel": "B", "match_id": figure1.PANEL_B_MATCH_ID, "phase": figure1.PANEL_B_MATCH_PHASE, "team": figure1.PANEL_B_TEAM, "source": figure1.PANEL_B_SOURCE_KEY, "t0": figure1.PANEL_B_T0},
    {"panel": "C", "match_id": figure1.PANEL_C_MATCH_ID, "phase": figure1.PANEL_C_MATCH_PHASE, "team": figure1.PANEL_C_TEAM, "source": figure1.PANEL_C_SOURCE_KEY, "t0": None},
])
display(selection)

## 4. Construct the publication figure

In [ ]:
fig = None
if BUILD_FIGURE and cache_ready:
    fig = figure1.main(close_figure=False)
elif BUILD_FIGURE:
    print("Processed trajectory/run/order cache unavailable; using frozen Figure 1.")

display_mode = show_figure(fig, FINAL_FIGURES / "figure1_transport_mechanism_schematic.png")
print("figure_display_mode", display_mode)